# পাঠ ০৯ - মেটাকগনিশন ডিজাইন প্যাটার্ন


## সেটআপ

এই নোটবুকটি Microsoft Agent Framework ব্যবহার করে Metacognition ডিজাইন প্যাটার্নটি প্রদর্শন করে।

**প্রয়োজনীয়তা:**
- পরিবেশ ভেরিয়েবল দ্বারা কনফিগার করা Azure OpenAI ডিপ্লয়মেন্ট
- Azure CLI প্রমাণীকৃত (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## মেটাকগনিশন কী?

মেটাকগনিশন হল **চিন্তা সম্পর্কে চিন্তা করা**। কৃত্রিম বুদ্ধিমত্তা এজেন্টদের প্রসঙ্গে, এর অর্থ হল এমন এজেন্ট তৈরি করা যা করতে পারে:

- **আত্ম-প্রতিফলন** তাদের নিজস্ব আউটপুট এবং যুক্তি প্রক্রিয়ার উপর
- **ত্রুটি সনাক্ত করা** এবং নিঃশব্দে ব্যর্থ হওয়ার বদলে শালীনভাবে পুনরুদ্ধার করা
- **মূল্যায়ন করা** যে তাদের উত্তরগুলি সম্পূর্ণ এবং সহায়ক কি না
- **অভিযোজিত করা** তাদের কৌশল যখন প্রাথমিক পন্থা কাজ করে না (উদাহরণস্বরূপ, একটি ব্যাকআপ সিস্টেমে ফিরে যাওয়া)

একটি মেটাকগনিটিভ এজেন্ট কেবল প্রশ্নগুলোর উত্তরই দেয় না — এটি নিজস্ব কর্মক্ষমতা পর্যবেক্ষণ করে এবং প্রয়োজনমতো তৎক্ষণাৎ সমন্বয় করে।


## প্রাথমিক এবং ব্যাকআপ টুল

A common metacognition pattern is the **ফলব্যাক কৌশল**। The agent tries a primary tool first; if it fails (e.g., a 404 error), the agent recognizes the failure and transparently switches to a backup tool.

This mirrors real-world systems where primary services may be unavailable and agents must self-diagnose the issue before choosing an alternative path.

Below we define two flight lookup tools:
- **প্রাথমিক** — কভার করে প্যারিস, টোকিও, এবং বার্সেলোনা
- **ব্যাকআপ** — কভার করে বার্লিন, সিডনি, এবং নিউ ইয়র্ক সিটি


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## ত্রুটি পুনরুদ্ধারের সঙ্গে আত্ম-প্রতিফলনকারী এজেন্ট

নীচের এজেন্টকে নির্দেশ দেওয়া হয়েছে প্রথমে প্রধান ফ্লাইট সিস্টেমটি চেষ্টা করতে, ব্যর্থতা শনাক্ত করতে, এবং স্বচ্ছভাবে ব্যাকআপ সিস্টেমে ফিরে যেতে। প্রতিটি প্রতিক্রিয়ার পরে এটি সংক্ষেপে নিজেই মূল্যায়ন করে যে এটি ব্যবহারকারীর প্রশ্নটি সম্পূর্ণভাবে উত্তর করেছে কি না।


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## স্ব-মূল্যায়ন প্যাটার্ন

মেটাকগনিশনের আরেকটি দিক হল **স্ব-মূল্যায়ন**: একটি পৃথক এজেন্ট (অথবা একই এজেন্ট দ্বিতীয় ধাপে) একটি প্রতিক্রিয়া সম্পূর্ণতা, যথার্থতা, এবং সহায়কতার জন্য পর্যালোচনা করে।

নিচে আমরা একটি `ResponseEvaluator` এজেন্ট তৈরি করব যা ভ্রমণ-এজেন্টের প্রতিক্রিয়াগুলিকে তিনটি মাত্রায় স্কোর করে।


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## সারাংশ

এই পাঠে আপনি শিখেছেন কীভাবে Microsoft Agent Framework ব্যবহার করে **মেটাকগনিটিভ এজেন্ট** তৈরি করতে:

- **স্ব-পর্যালোচনা**: এজেন্টগুলি তাদের নিজস্ব যুক্তি পর্যবেক্ষণ করে এবং কি ঘটেছিল তা স্বচ্ছভাবে যোগাযোগ করে।
- **ফলব্যাকসহ ত্রুটি পুনরুদ্ধার**: একটি প্রধান + ব্যাকআপ টুল প্যাটার্ন যেখানে এজেন্ট ব্যর্থতা সনাক্ত করে (যেমন, 404 ত্রুটি) এবং স্বয়ংক্রিয়ভাবে বিকল্প উৎস চেষ্টা করে।
- **স্ব-মূল্যায়ন**: একটি আলাদা মূল্যায়ক এজেন্ট যা প্রত্যুত্তরগুলোকে সম্পূর্ণতা, সঠিকতা, এবং সহায়কতার জন্য স্কোর করে।

এই প্যাটার্নগুলো এজেন্টদের আরও স্থিতিশীল, স্বচ্ছ, এবং বিশ্বাসযোগ্য করে তোলে — প্রোডাকশনে মোতায়েনের জন্য অত্যন্ত গুরুত্বপূর্ণ গুণাবলী।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
দায়-অস্বীকার:
এই নথিটি AI অনুবাদ সেবা Co-op Translator (https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত করা হয়েছে। যদিও আমরা যথাসাধ্য নির্ভুলতার চেষ্টা করি, অনুগ্রহ করে মনে রাখুন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার নিজ ভাষায়ই প্রামাণিক উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের ক্ষেত্রে পেশাদার মানব অনুবাদ গ্রহণ করার পরামর্শ দেওয়া হয়। এই অনুবাদের ব্যবহারের ফলে যে কোনো ভুলবোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়ী নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
